# Create CSV files from the split folders

This notebook reads the three split folders and creates one CSV file for each split of each task (9 files in total).

Each CSV file has two columns: `image_id` and the class name.

`split_real_fake_*.csv`

| image_id | label |
|---|---|
| 1 | real |
| 3 | fake |

`split_age_*.csv`

| image_id | age |
|---|---|
| 1 | 26-35 |
| 4 | under_18 |

`split_gender_*.csv`

| image_id | gender |
|---|---|
| 1 | female |
| 3 | unknown |

**Output files** (saved in the `data` folder):

- `split_real_fake_train.csv`, `split_real_fake_val.csv`, `split_real_fake_test.csv`
- `split_age_train.csv`, `split_age_val.csv`, `split_age_test.csv`
- `split_gender_train.csv`, `split_gender_val.csv`, `split_gender_test.csv`

**How it works**

- The image id is the file name without the extension. For example, `1246.jpg` becomes `1246`.
- The class is the name of the folder that contains the image. For example, `split_age/train/under_18/55.jpg` has the class `under_18`.
- The notebook reads the folders directly, so images you added by hand are included too.

In [1]:
import pandas as pd
from pathlib import Path

## Step 1: Set the paths and column names

In [2]:
DATA_DIR = Path(r"C:\Users\jagad\Documents\my_classes\Tasks\my_projects\Deep_Fake_project_final\data")

OUT_DIR = DATA_DIR                 # the CSV files are saved here
SPLITS = ["train", "val", "test"]
IMAGE_TYPES = {".jpg", ".jpeg", ".png"}

# folder name -> (class column name in the CSV, class folders we expect to find)
TASKS = {
    "split_age":       ("age",    ["18-25", "26-35", "36-50", "50+", "not_applicable", "under_18"]),
    "split_gender":    ("gender", ["female", "male", "unknown"]),
    "split_real_fake": ("label",  ["real", "fake"]),
}

for folder in TASKS:
    if not (DATA_DIR / folder).exists():
        raise FileNotFoundError("Folder not found: " + str(DATA_DIR / folder))

## Step 2: Read one split folder

This function opens the class folders inside one split (for example `split_age/train/`) and collects the image id and class name of every image.

In [3]:
def scan_split(task_folder, split):
    root = DATA_DIR / task_folder / split
    rows = []
    for class_folder in sorted(root.iterdir()):
        if not class_folder.is_dir():
            continue
        for image in class_folder.iterdir():
            if image.suffix.lower() in IMAGE_TYPES:          # skip other files such as Thumbs.db
                rows.append({"image_id": image.stem, "label": class_folder.name})
    return pd.DataFrame(rows, columns=["image_id", "label"])

## Step 3: Create the CSV files

For each task and each split, this step creates the CSV file and runs two checks:

- A warning is printed if a class folder has an unexpected name (for example `under 18` instead of `under_18`) or if an expected class folder is missing.
- A warning is printed if the same `image_id` appears more than once.

In [4]:
def make_csv(task_folder, column_name, expected_classes, split):
    df = scan_split(task_folder, split)

    # check the class folder names
    unexpected = sorted(set(df["label"]) - set(expected_classes))
    missing = [c for c in expected_classes if c not in set(df["label"])]
    if unexpected:
        print("   WARNING: unexpected class folders found:", unexpected)
    if missing:
        print("   WARNING: class folders not found:", missing)

    out = df.rename(columns={"label": column_name})          # columns: image_id, <class column>

    # sort by image id (numbers first, then text ids)
    out["sort_key"] = pd.to_numeric(out["image_id"], errors="coerce")
    out = out.sort_values(["sort_key", "image_id"]).drop(columns="sort_key").reset_index(drop=True)

    # check for repeated image ids (should be 0)
    repeated = out["image_id"].duplicated().sum()
    if repeated > 0:
        print("   WARNING: repeated image_id count:", repeated)

    file_path = OUT_DIR / f"{task_folder}_{split}.csv"
    out.to_csv(file_path, index=False)
    print(f"{file_path.name:28s} rows = {len(out):5d}")
    return out


results = {}
for task_folder, (column_name, expected_classes) in TASKS.items():
    for split in SPLITS:
        results[(task_folder, split)] = make_csv(task_folder, column_name, expected_classes, split)

split_age_train.csv          rows =  3561
split_age_val.csv            rows =   763
split_age_test.csv           rows =   764
split_gender_train.csv       rows =  3561
split_gender_val.csv         rows =   763
split_gender_test.csv        rows =   764
split_real_fake_train.csv    rows =  3561
split_real_fake_val.csv      rows =   763
split_real_fake_test.csv     rows =   764


## Step 4: Check the number of images in each class

In [5]:
for task_folder, (column_name, expected_classes) in TASKS.items():
    summary = pd.DataFrame({split: results[(task_folder, split)][column_name].value_counts() for split in SPLITS}).fillna(0).astype(int)
    summary.loc["TOTAL"] = summary.sum()
    print("=====", task_folder, "=====")
    print(summary, "\n")

===== split_age =====
                train  val  test
age                             
18-25             254   26    50
26-35            1089  160   235
36-50             223   59    56
50+               102   98    19
not_applicable   1844  393   401
under_18           49   27     3
TOTAL            3561  763   764 

===== split_gender =====
         train  val  test
gender                   
unknown   1836  383   393
female    1001  228   224
male       724  152   147
TOTAL     3561  763   764 

===== split_real_fake =====
       train  val  test
label                  
real    1812  390   391
fake    1749  373   373
TOTAL   3561  763   764 



## Step 5: Look at one file

In [6]:
# first 10 rows of the age training CSV
results[("split_age", "train")].head(10)

,image_id,age
0,1,26-35
1,2,under_18
2,6,18-25
3,7,50+
4,8,18-25
5,9,36-50
6,10,not_applicable
7,11,36-50
8,12,36-50
9,13,18-25
